In [1]:
import pandas as pd
import networkx as nx

# 1. قراءة ملفات الـ GTFS الأساسية
stops_df = pd.read_csv('stops.txt')
stop_times_df = pd.read_csv('stop_times.txt')

print(f"تم تحميل عدد {len(stops_df)} محطة بنجاح.")

# 2. بناء شبكة المترو (Graph)
metro_graph = nx.Graph()

# إضافة كل محطة كعقدة (Node) مع حفظ اسمها وإحداثياتها
for _, row in stops_df.iterrows():
    metro_graph.add_node(
        row['stop_id'], 
        name=row['stop_name'], 
        lat=row['stop_lat'], 
        lon=row['stop_lon']
    )

# 3. ربط المحطات ببعضها بناءً على الجدول الزمني للرحلات (Trips)
for i in range(len(stop_times_df) - 1):
    current_row = stop_times_df.iloc[i]
    next_row = stop_times_df.iloc[i+1]
    
    # التأكد من أن المحطتين تتبعان نفس الرحلة لضمان التتابع السليم
    if current_row['trip_id'] == next_row['trip_id']:
        u = current_row['stop_id']
        v = next_row['stop_id']
        # إضافة الضلع بوزن افتراضي (دقيقة واحدة كمثال)
        metro_graph.add_edge(u, v, weight=1)

# 4. دالة البحث عن أسرع/أقصر مسار بين محطتين باستخدام معرفات المحطات (Stop IDs)
def find_route_by_id(start_id, end_id):
    try:
        # استخدام خوارزمية Dijkstra الشهيرة لحساب أقصر مسار في NetworkX
        path_ids = nx.shortest_path(metro_graph, source=start_id, target=end_id, weight='weight')
        
        # تحويل الـ IDs إلى أسماء المحطات الحقيقية المعروضة للمستخدم
        path_names = [metro_graph.nodes[node]['name'] for node in path_ids]
        
        return path_names
    except nx.NetworkXNoPath:
        return "عذراً، لا يوجد مسار متاح بين هاتين المحطتين."
    except KeyError as e:
        return f"خطأ في معرف المحطة غير الموجود: {e}"

print("تم بناء محرك المسارات بنجاح وجاهز للاستخدام!")

تم تحميل عدد 189 محطة بنجاح.
تم بناء محرك المسارات بنجاح وجاهز للاستخدام!


In [2]:
# 1. اعرض أول 10 محطات عشان تاخد منهم الـ stop_id الحقيقي للتجربة
print("عينة من المحطات وأكوادها (Stop IDs):")
display(stops_df[['stop_id', 'stop_name']].head(10))

# 2. جرب تبحث عن مسار (استبدل القيم دي بـ stop_id حقيقي ظاهر قدامك في الجدول)
# مثلاً لو محطة رقم '1' ومحطة رقم '5'
start_station = stops_df['stop_id'].iloc[0]  # أول محطة كمثال
end_station = stops_df['stop_id'].iloc[5]    # المحطة السادسة كمثال

route_result = find_route_by_id(start_station, end_station)

print(f"\nالنتيجة من المحطة {start_station} إلى {end_station}:")
print(route_result)

عينة من المحطات وأكوادها (Stop IDs):


,stop_id,stop_name
0,10_HLW_METRO_N,Helwan
1,10_AHL_METRO_N,Ain Helwan
2,10_UHL_METRO_N,Helwan University
3,14_HOF_METRO_N,Wadi Hof
4,14_HHL_METRO_N,Hadayek Helwan
5,7_MAA_METRO_N,El-Maasara
6,27_TAS_METRO_N,Tora El-Asmant
7,26_KZK_METRO_N,Kozzika
8,28_TBA_METRO_N,Tora El-Balad
9,23_SMA_METRO_N,Sakanat El-Maadi



النتيجة من المحطة 10_HLW_METRO_N إلى 7_MAA_METRO_N:
['Helwan', 'Ain Helwan', 'Helwan University', 'Wadi Hof', 'Hadayek Helwan', 'El-Maasara']


In [3]:
def find_route_by_name(start_name, end_name):
    try:
        # البحث عن الـ stop_id بناءً على اسم المحطة
        start_row = stops_df[stops_df['stop_name'].str.strip() == start_name]
        end_row = stops_df[stops_df['stop_name'].str.strip() == end_name]
        
        if start_row.empty or end_row.empty:
            return "عذراً، إحدى المحطات غير موجودة في النظام. تأكد من كتابة الاسم صحيحاً."
            
        start_id = start_row['stop_id'].values[0]
        end_id = end_row['stop_id'].values[0]
        
        # حساب المسار باستخدام الـ IDs
        path_ids = nx.shortest_path(metro_graph, source=start_id, target=end_id, weight='weight')
        
        # تحويل الـ IDs إلى أسماء المحطات لعرضها للمستخدم
        path_names = [metro_graph.nodes[node]['name'] for node in path_ids]
        
        return path_names
    except nx.NetworkXNoPath:
        return "لا يوجد مسار متاح بين هاتين المحطتين."

# تجربة البحث بالأسماء الحقيقية
# (تأكد من اختيار اسمين موجودين في ملف الـ stops)
sample_route = find_route_by_name('Helwan', 'El-Maasara')
print("مسار الرحلة بالأسماء:", sample_route)

مسار الرحلة بالأسماء: ['Helwan', 'Ain Helwan', 'Helwan University', 'Wadi Hof', 'Hadayek Helwan', 'El-Maasara']


In [4]:
pip install fastapi uvicorn

In [ ]:
# ربط المستودع المحلي بمستودع GitHub (استبدل الرابط برابط مستودعك الحقيقي)
git remote add origin https://github.com/abdarhmanali692/MetroVision.git

# تغيير اسم الفرع الأساسي إلى main
git branch -M main

# رفع الكود على GitHub
git push -u origin main